In [17]:
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [18]:
#from google.colab import drive
#drive.mount('/content/drive')


In [19]:
base_dir = "/Users/yub/Documents/ssu_32/CapstoneDesign1/Code"
print("MyDrive 루트:", base_dir)
print(os.listdir(base_dir))


MyDrive 루트: /Users/yub/Documents/ssu_32/CapstoneDesign1/Code
['캡스톤_전처리_수정.ipynb', '.DS_Store', 'LSTM.ipynb', 'LSTM_tf.ipynb', 'One-Class_SVM.ipynb', 'lstm_flow_nids_tf.py', 'Isolation-Forest.ipynb', 'flow_preprocessed_final.csv', 'TCN.ipynb']


In [20]:
NF_CSV_PATH = "/Users/yub/Documents/ssu_32/CapstoneDesign1/Datasets/NF-UNSW-NB15-v3.csv"

feature_columns = [
    "L4_SRC_PORT", "L4_DST_PORT",
    "PROTOCOL","IN_BYTES","IN_PKTS","OUT_BYTES","OUT_PKTS",
    "TCP_FLAGS","CLIENT_TCP_FLAGS","SERVER_TCP_FLAGS",
    "FLOW_DURATION_MILLISECONDS","DURATION_IN","DURATION_OUT",
    "MIN_TTL","MAX_TTL","LONGEST_FLOW_PKT","SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN","MAX_IP_PKT_LEN",
    "SRC_TO_DST_SECOND_BYTES","DST_TO_SRC_SECOND_BYTES",
    "RETRANSMITTED_IN_BYTES","RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES","RETRANSMITTED_OUT_PKTS",
    "SRC_TO_DST_AVG_THROUGHPUT","DST_TO_SRC_AVG_THROUGHPUT",
    "NUM_PKTS_UP_TO_128_BYTES","NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES","NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES",
    "TCP_WIN_MAX_IN","TCP_WIN_MAX_OUT",
    "ICMP_TYPE","ICMP_IPV4_TYPE",
    "DNS_QUERY_ID","DNS_QUERY_TYPE","DNS_TTL_ANSWER",
    "FTP_COMMAND_RET_CODE",
    "SRC_TO_DST_IAT_MIN","SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_AVG","SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN","DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_AVG","DST_TO_SRC_IAT_STDDEV",
]

if not os.path.exists(NF_CSV_PATH):
    raise FileNotFoundError(f"UNSW 파일을 찾을 수 없습니다: {NF_CSV_PATH}")

all_column_names = feature_columns + ['attack_cat', 'Label', 'additional_label_col']

df_raw = pd.read_csv(NF_CSV_PATH, header=None, names=all_column_names)
print("원본 데이터 크기:", df_raw.shape)
df_raw.head()

/var/folders/53/y79xp_3s02b7prv0r89jnlq00000gn/T/ipykernel_79776/3913557460.py:32: DtypeWarning: Columns (0,1,3,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(NF_CSV_PATH, header=None, names=all_column_names)


원본 데이터 크기: (2365425, 51)


,,,,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,...,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,attack_cat,Label,additional_label_col
FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,178,2,0,0,...,0,0,0,0,0,0,0,0,0,Benign
1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,2976,28,27,27,...,0,91,12,19,0,90,12,19,0,Benign
1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,548216,438,27,27,...,0,1843,10,119,0,1843,5,88,0,Benign
1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,178,2,0,0,...,0,0,0,0,0,0,0,0,0,Benign


In [21]:
# feature_columns 정의는 이제 이전 셀(csdg5N4vOzRJ)로 이동되었습니다.

print("원본 컬럼 개수:", len(df_raw.columns))
print("예시 컬럼들:", list(df_raw.columns[:20]))

원본 컬럼 개수: 51
예시 컬럼들: ['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES']


In [22]:
df = df_raw.copy()

print("라벨 관련 컬럼들:")
for col in df.columns:
    if col.lower().startswith("label") or "attack" in col.lower():
        print("  -", col)

if "Label" in df.columns and df["Label"].nunique() <= 2 and df["Label"].notna().all():
    print("\n[INFO] 기존 Label 컬럼(0/1)을 그대로 사용합니다.")
    label_series = df["Label"].astype(int)

elif "Attack" in df.columns:
    print("\n[INFO] Attack 컬럼 기준으로 Label(0/1)을 생성합니다. (Normal=0, 나머지=1)")
    label_series = (df["Attack"] != "Normal").astype(int)

elif "attack_cat" in df.columns:
    print("\n[INFO] attack_cat 컬럼 기준으로 Label(0/1)을 생성합니다. (0=정상, 나머지=1)")
    label_series = (df["attack_cat"] != 0).astype(int)

else:
    raise ValueError("어떤 컬럼을 기준으로 정상/공격을 나눠야 할지 모르겠습니다. Label/Attack/attack_cat 중 하나가 필요합니다.")

df["Label"] = label_series

print("\nLabel 분포 (0=정상, 1=공격):")
print(df["Label"].value_counts())


라벨 관련 컬럼들:
  - attack_cat
  - Label

[INFO] attack_cat 컬럼 기준으로 Label(0/1)을 생성합니다. (0=정상, 나머지=1)

Label 분포 (0=정상, 1=공격):
Label
1    1302903
0    1062522
Name: count, dtype: int64


In [23]:

final_columns = feature_columns + ["Label"]

missing = [c for c in final_columns if c not in df.columns]
if missing:
    print("[WARNING] 데이터에 없는 컬럼들이 있습니다:", missing)

df_final = df[final_columns].copy()
print("최종 데이터 크기:", df_final.shape)

OUTPUT_CSV_PATH = "/Users/yub/Documents/ssu_32/CapstoneDesign1/Code/flow_preprocessed_final.csv"

os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
df_final.to_csv(OUTPUT_CSV_PATH, index=False)

print("전처리 CSV 저장 완료:", OUTPUT_CSV_PATH)


최종 데이터 크기: (2365425, 49)
전처리 CSV 저장 완료: /Users/yub/Documents/ssu_32/CapstoneDesign1/Code/flow_preprocessed_final.csv


In [24]:
import pandas as pd
import os

OUTPUT_CSV_PATH = "/Users/yub/Documents/ssu_32/CapstoneDesign1/Code/flow_preprocessed_final.csv"

feature_columns = [
    "L4_SRC_PORT", "L4_DST_PORT",
    "PROTOCOL","IN_BYTES","IN_PKTS","OUT_BYTES","OUT_PKTS",
    "TCP_FLAGS","CLIENT_TCP_FLAGS","SERVER_TCP_FLAGS",
    "FLOW_DURATION_MILLISECONDS","DURATION_IN","DURATION_OUT",
    "MIN_TTL","MAX_TTL","LONGEST_FLOW_PKT","SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN","MAX_IP_PKT_LEN",
    "SRC_TO_DST_SECOND_BYTES","DST_TO_SRC_SECOND_BYTES",
    "RETRANSMITTED_IN_BYTES","RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES","RETRANSMITTED_OUT_PKTS",
    "SRC_TO_DST_AVG_THROUGHPUT","DST_TO_SRC_AVG_THROUGHPUT",
    "NUM_PKTS_UP_TO_128_BYTES","NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES","NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES",
    "TCP_WIN_MAX_IN","TCP_WIN_MAX_OUT",
    "ICMP_TYPE","ICMP_IPV4_TYPE",
    "DNS_QUERY_ID","DNS_QUERY_TYPE","DNS_TTL_ANSWER",
    "FTP_COMMAND_RET_CODE",
    "SRC_TO_DST_IAT_MIN","SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_AVG","SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN","DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_AVG","DST_TO_SRC_IAT_STDDEV",
]

df_iso = pd.read_csv(OUTPUT_CSV_PATH)
print("Isolation Forest용 데이터 크기:", df_iso.shape)

# 1) feature만 추출
X = df_iso[feature_columns].copy()

# 2) TCP_FLAGS 계열이 혹시 문자열(0x..)이면 숫자로 변환
for col in ["TCP_FLAGS", "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS"]:
    if col in X.columns and X[col].dtype == "object":
        X[col] = X[col].apply(
            lambda v: int(str(v), 16) if isinstance(v, (str, bytes)) and str(v).startswith("0x") else v
        )
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

# 3) PROTOCOL 같은 문자열 컬럼 처리
for col in X.columns:
    if X[col].dtype == "object":
        if col == "PROTOCOL":
            # tcp/udp/icmp 같은 건 카테고리 코드로
            X[col] = X[col].astype("category").cat.codes
        else:
            # 나머지는 숫자로 변환 시도
            X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

# 4) float32로 줄여서 메모리 절약
X = X.astype("float32")

LABEL_COLUMN = "Label"
NORMAL_LABEL_VALUE = 0
y = df_iso[LABEL_COLUMN].astype(int)

print("전처리된 X 데이터 크기:", X.shape)
print("Label 분포 (0=정상, 1=공격):")
print(y.value_counts())


/var/folders/53/y79xp_3s02b7prv0r89jnlq00000gn/T/ipykernel_79776/318953808.py:30: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df_iso = pd.read_csv(OUTPUT_CSV_PATH)


Isolation Forest용 데이터 크기: (2365425, 49)
전처리된 X 데이터 크기: (2365425, 48)
Label 분포 (0=정상, 1=공격):
Label
1    1302903
0    1062522
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split

TRAIN_NORMAL_RATIO = 0.8

normal_mask = (y == NORMAL_LABEL_VALUE)

X_normal = X[normal_mask]
y_normal = y[normal_mask]

X_train, X_val = train_test_split(
    X_normal,
    train_size=TRAIN_NORMAL_RATIO,
    random_state=42,
    shuffle=True
)

X_test = X
y_test = y

print("Train(Normal):", X_train.shape)
print("Val(Normal):  ", X_val.shape)
print("Test(All):    ", X_test.shape)


Train(Normal): (850017, 48)
Val(Normal):   (212505, 48)
Test(All):     (2365425, 48)


In [26]:
from sklearn.ensemble import IsolationForest

IF_PARAMS = {
    "n_estimators": 200,
    "max_samples": "auto",
    "contamination": 0.05,
    "random_state": 42,
    "n_jobs": -1
}

iso_model = IsolationForest(**IF_PARAMS)
iso_model.fit(X_train)

print("Isolation Forest 학습 완료")

Isolation Forest 학습 완료


In [27]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_raw = iso_model.predict(X_test)
y_pred = (y_pred_raw == -1).astype(int)

print("=== Confusion Matrix (0=정상,1=공격) ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))

=== Confusion Matrix (0=정상,1=공격) ===
[[1009455   53067]
 [ 333823  969080]]

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.7515    0.9501    0.8392   1062522
           1     0.9481    0.7438    0.8336   1302903

    accuracy                         0.8364   2365425
   macro avg     0.8498    0.8469    0.8364   2365425
weighted avg     0.8598    0.8364    0.8361   2365425



In [28]:
import numpy as np

scores = iso_model.decision_function(X_test)
percentile = 5.0
threshold = np.percentile(scores, percentile)
print("score range:", scores.min(), "~", scores.max())
print("사용 threshold:", threshold)

y_pred_thr = (scores < threshold).astype(int)

print("\n=== Confusion Matrix (Custom Threshold) ===")
print(confusion_matrix(y_test, y_pred_thr))

print("\n=== Classification Report (Custom Threshold) ===")
print(classification_report(y_test, y_pred_thr, digits=4))

score range: -0.22725204667521226 ~ 0.18846614063916028
사용 threshold: -0.1407778917008787

=== Confusion Matrix (Custom Threshold) ===
[[1059016    3506]
 [1188137  114766]]

=== Classification Report (Custom Threshold) ===
              precision    recall  f1-score   support

           0     0.4713    0.9967    0.6400   1062522
           1     0.9704    0.0881    0.1615   1302903

    accuracy                         0.4962   2365425
   macro avg     0.7208    0.5424    0.4007   2365425
weighted avg     0.7462    0.4962    0.3764   2365425



In [29]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 반복 횟수 (TCN에서 epoch처럼)
N_RUNS = 10

train_losses = []
val_losses = []
test_accs = []
test_f1_attacks = []

for epoch in range(N_RUNS):
    # 매 에폭마다 random_state만 살짝 바꿔서 다른 숲 학습
    rs = 42 + epoch

    iso_model = IsolationForest(
        n_estimators=200,
        max_samples="auto",
        contamination=0.05,
        random_state=rs,
        n_jobs=-1
    )

    # ===== 학습 =====
    iso_model.fit(X_train)

    # ===== train/val "loss" 계산 =====
    # IsolationForest는 손실이 없으니까
    # 편의상 -mean(decision_function) 을 loss처럼 사용
    train_scores = iso_model.decision_function(X_train)  # 값이 클수록 정상
    val_scores = iso_model.decision_function(X_val)

    train_loss = -train_scores.mean()
    val_loss = -val_scores.mean()

    # ===== test 성능 계산 =====
    y_pred_raw = iso_model.predict(X_test)        # 1=정상, -1=이상
    y_pred = (y_pred_raw == -1).astype(int)       # 1=공격, 0=정상

    acc = accuracy_score(y_test, y_pred)
    f1_attack = f1_score(y_test, y_pred, pos_label=1)
    prec_attack = precision_score(y_test, y_pred, pos_label=1)
    rec_attack = recall_score(y_test, y_pred, pos_label=1)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    test_accs.append(acc)
    test_f1_attacks.append(f1_attack)

    print(
        f"[Epoch {epoch+1:02d}] "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  "
        f"acc={acc:.4f}  "
        f"prec(attack)={prec_attack:.4f}  "
        f"recall(attack)={rec_attack:.4f}  "
        f"f1(attack)={f1_attack:.4f}"
    )


[Epoch 01] train_loss=-0.1156  val_loss=-0.1157  acc=0.8364  prec(attack)=0.9481  recall(attack)=0.7438  f1(attack)=0.8336
[Epoch 02] train_loss=-0.1217  val_loss=-0.1217  acc=0.8011  prec(attack)=0.9434  recall(attack)=0.6797  f1(attack)=0.7901
[Epoch 03] train_loss=-0.1253  val_loss=-0.1253  acc=0.8415  prec(attack)=0.9486  recall(attack)=0.7530  f1(attack)=0.8396
[Epoch 04] train_loss=-0.1147  val_loss=-0.1148  acc=0.8685  prec(attack)=0.9516  recall(attack)=0.8021  f1(attack)=0.8704
[Epoch 05] train_loss=-0.1192  val_loss=-0.1192  acc=0.8519  prec(attack)=0.9498  recall(attack)=0.7718  f1(attack)=0.8516
[Epoch 06] train_loss=-0.1235  val_loss=-0.1236  acc=0.8547  prec(attack)=0.9502  recall(attack)=0.7770  f1(attack)=0.8549
[Epoch 07] train_loss=-0.1150  val_loss=-0.1151  acc=0.8329  prec(attack)=0.9476  recall(attack)=0.7373  f1(attack)=0.8293
[Epoch 08] train_loss=-0.1163  val_loss=-0.1163  acc=0.8592  prec(attack)=0.9508  recall(attack)=0.7850  f1(attack)=0.8600
[Epoch 09] train